# 🏠 House Price Prediction — Regression Analysis
**Dataset:** Ames Housing Dataset — residential property sales with 16 features  
**Goal:** Predict house sale prices using regression models and understand what drives property value  
**Author:** Arman Arabkhani | AUT Data Science


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette(['steelblue'])
print("Libraries loaded successfully.")

## 2. Data Loading & Overview
We use the Ames Housing dataset — a rich dataset of residential property sales containing features such as size, quality, age, and neighbourhood.


In [ ]:
# Load dataset
df = pd.read_csv('ames_housing.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures: {list(df.columns)}")
df.head()

In [ ]:
# Basic info
df.info()

In [ ]:
# Summary statistics
df.describe()

## 3. Handling Missing Values
Before building models, we need to identify and handle missing values appropriately.


In [ ]:
# Check missing values
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing)

# Visualise missing values
if len(missing) > 0:
    plt.figure(figsize=(8, 4))
    missing.plot(kind='bar', color='steelblue', edgecolor='white')
    plt.title('Missing Values by Column', fontsize=14)
    plt.xlabel('Column')
    plt.ylabel('Missing Count')
    plt.tight_layout()
    plt.show()

In [ ]:
# Fill numeric missing values with median (robust to outliers)
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Fill categorical missing values with mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f"Missing values after imputation: {df.isnull().sum().sum()}")

## 4. Exploratory Data Analysis (EDA)
### 4.1 Target Variable — Sale Price Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution
axes[0].hist(df['SalePrice'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('SalePrice Distribution', fontsize=13)
axes[0].set_xlabel('Sale Price ($)')
axes[0].set_ylabel('Count')

# Log-transformed (often better for regression)
axes[1].hist(np.log1p(df['SalePrice']), bins=40, color='coral', edgecolor='white')
axes[1].set_title('Log(SalePrice) Distribution', fontsize=13)
axes[1].set_xlabel('Log Sale Price')
axes[1].set_ylabel('Count')

plt.suptitle('Sale Price: Raw vs Log-Transformed', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Mean price:   ${df['SalePrice'].mean():,.0f}")
print(f"Median price: ${df['SalePrice'].median():,.0f}")
print(f"Price range:  ${df['SalePrice'].min():,.0f} — ${df['SalePrice'].max():,.0f}")

### 4.2 Key Feature Relationships

In [ ]:
# Top numeric features vs SalePrice
num_features = ['Overall_Qual', 'Gr_Liv_Area', 'Garage_Cars', 'Total_Bsmt_SF', 'Year_Built', 'Full_Bath']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, feat in zip(axes.flatten(), num_features):
    ax.scatter(df[feat], df['SalePrice'], alpha=0.4, color='steelblue', s=15, edgecolors='white', linewidth=0.3)
    m, b = np.polyfit(df[feat].dropna(), df['SalePrice'][df[feat].notna()], 1)
    x_line = np.linspace(df[feat].min(), df[feat].max(), 100)
    ax.plot(x_line, m * x_line + b, 'r--', linewidth=1.5, label='Trend')
    ax.set_xlabel(feat)
    ax.set_ylabel('Sale Price ($)')
    ax.set_title(f'{feat} vs SalePrice')
    ax.legend(fontsize=8)

plt.suptitle('Key Features vs Sale Price', fontsize=14)
plt.tight_layout()
plt.show()

### 4.3 Categorical Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['Overall_Qual', 'Exter_Qual', 'Kitchen_Qual']):
    if df[col].dtype == 'object' or df[col].nunique() < 12:
        grp = df.groupby(col)['SalePrice'].median().sort_values(ascending=False)
        ax.bar(grp.index.astype(str), grp.values, color='steelblue', edgecolor='white')
        ax.set_title(f'Median Price by {col}', fontsize=12)
        ax.set_xlabel(col)
        ax.set_ylabel('Median Sale Price ($)')
        ax.tick_params(axis='x', rotation=30)

plt.suptitle('Median Sale Price by Categorical Features', fontsize=14)
plt.tight_layout()
plt.show()

### 4.4 Correlation Heatmap

In [ ]:
# Correlation matrix for numeric features
corr_cols = ['SalePrice', 'Overall_Qual', 'Gr_Liv_Area', 'Garage_Cars',
             'Total_Bsmt_SF', 'Year_Built', 'Full_Bath', 'TotRms_AbvGrd',
             'Fireplaces', 'Lot_Area']

corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap — Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

print("\nTop correlations with SalePrice:")
print(corr['SalePrice'].sort_values(ascending=False)[1:])

## 5. Feature Engineering
Creating new meaningful features from existing ones can significantly improve model performance.


In [ ]:
df_model = df.copy()

# Age of house at time of sale
df_model['House_Age'] = 2010 - df_model['Year_Built']

# Years since remodel
df_model['Years_Since_Remod'] = 2010 - df_model['Year_Remod_Add']

# Total living area (basement + above ground)
df_model['Total_SF'] = df_model['Total_Bsmt_SF'] + df_model['Gr_Liv_Area']

# Has garage
df_model['Has_Garage'] = (df_model['Garage_Cars'] > 0).astype(int)

# Has fireplace
df_model['Has_Fireplace'] = (df_model['Fireplaces'] > 0).astype(int)

# Has central air
df_model['Has_CentralAir'] = (df_model['Central_Air'] == 'Y').astype(int)

# Encode quality ratings
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1}
for col in ['Exter_Qual', 'Kitchen_Qual']:
    df_model[col + '_Enc'] = df_model[col].map(qual_map).fillna(3)

# Encode other categoricals
le = LabelEncoder()
for col in ['Neighborhood', 'Bldg_Type', 'House_Style']:
    df_model[col + '_Enc'] = le.fit_transform(df_model[col].astype(str))

print("New features created:")
new_feats = ['House_Age', 'Years_Since_Remod', 'Total_SF', 'Has_Garage', 'Has_Fireplace', 'Has_CentralAir']
print(df_model[new_feats].head())

## 6. Model Building
We'll train and compare multiple regression models to find the best predictor of house prices.


In [ ]:
# Select features for modelling
feature_cols = [
    'Overall_Qual', 'Gr_Liv_Area', 'Garage_Cars', 'Total_Bsmt_SF',
    'Full_Bath', 'TotRms_AbvGrd', 'Fireplaces', 'Lot_Area',
    'House_Age', 'Years_Since_Remod', 'Total_SF',
    'Has_Garage', 'Has_Fireplace', 'Has_CentralAir',
    'Exter_Qual_Enc', 'Kitchen_Qual_Enc',
    'Neighborhood_Enc', 'Bldg_Type_Enc', 'House_Style_Enc'
]

X = df_model[feature_cols]
y = df_model['SalePrice']

# Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")
print(f"Features used: {len(feature_cols)}")

In [ ]:
# Train multiple models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=10),
    'Lasso Regression':  Lasso(alpha=100),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
}

results = {}
for name, model in models.items():
    if name in ['Linear Regression', 'Ridge Regression', 'Lasso Regression']:
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2   = r2_score(y_test, preds)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2, 'preds': preds}
    print(f"{name:<25} | MAE: ${mae:>8,.0f} | RMSE: ${rmse:>8,.0f} | R²: {r2:.4f}")

## 7. Model Evaluation & Comparison

In [ ]:
# Compare R² scores
names = list(models.keys())
r2_scores = [results[n]['R²'] for n in names]
mae_scores = [results[n]['MAE'] for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#c0c0c0', '#a0b4c8', '#7094b4', '#4074a0', '#1a5490']
axes[0].barh(names, r2_scores, color=colors, edgecolor='white')
axes[0].set_xlabel('R² Score')
axes[0].set_title('R² Score by Model (higher = better)', fontsize=12)
axes[0].set_xlim(0, 1)
for i, v in enumerate(r2_scores):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)

axes[1].barh(names, mae_scores, color=colors, edgecolor='white')
axes[1].set_xlabel('Mean Absolute Error ($)')
axes[1].set_title('MAE by Model (lower = better)', fontsize=12)
for i, v in enumerate(mae_scores):
    axes[1].text(v + 100, i, f'${v:,.0f}', va='center', fontsize=9)

plt.suptitle('Model Comparison', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Best model: Actual vs Predicted
best_model_name = max(results, key=lambda n: results[n]['R²'])
best_preds = results[best_model_name]['preds']

print(f"Best model: {best_model_name} (R² = {results[best_model_name]['R²']:.4f})")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, best_preds, alpha=0.4, color='steelblue', s=20, edgecolors='white', linewidth=0.3)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'{best_model_name} — Actual vs Predicted', fontsize=12)
axes[0].legend()

# Residuals
residuals = y_test - best_preds
axes[1].scatter(best_preds, residuals, alpha=0.4, color='coral', s=20, edgecolors='white', linewidth=0.3)
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted Price ($)')
axes[1].set_ylabel('Residual ($)')
axes[1].set_title('Residual Plot', fontsize=12)

plt.suptitle(f'Best Model Evaluation — {best_model_name}', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Feature Importance

In [ ]:
# Use Random Forest for feature importance
rf = models['Random Forest']
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
importances.head(12).plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Top 12 Most Important Features — Random Forest', fontsize=13)
plt.xlabel('Feature')
plt.ylabel('Importance Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("\nTop 5 features driving house prices:")
for feat, imp in importances.head(5).items():
    print(f"  {feat:<25} {imp:.4f}")

## 9. Summary & Conclusions

### Key Findings

**Best Performing Model:**
- Gradient Boosting and Random Forest significantly outperform linear models, capturing non-linear relationships in the data.
- Linear and Ridge Regression provide a solid baseline but miss complex feature interactions.

**Most Important Features:**
- **Overall Quality** is the single strongest predictor of house price — a well-built, high-quality home commands a premium regardless of size.
- **Total Square Footage** (above ground + basement) is the second strongest driver — buyers pay for usable space.
- **House Age** negatively impacts price — newer homes are significantly more valuable.
- **Garage capacity** adds measurable value, reflecting lifestyle preferences.

**Feature Engineering Impact:**
- Derived features like `Total_SF`, `House_Age`, and quality encodings improved model performance meaningfully over raw features alone.

**Practical Takeaways:**
- For a seller: investing in quality finishes and renovations yields the highest return.
- For a buyer: older homes in good neighbourhoods offer the best value relative to size.
- For a data scientist: ensemble methods (Random Forest, Gradient Boosting) are well-suited for structured tabular data with mixed feature types.

### What I Would Do Next
- Tune hyperparameters using GridSearchCV for further performance gains
- Experiment with log-transforming SalePrice (reduces skew, often improves linear model performance)
- Add interaction features (e.g. Quality × Size)
- Try XGBoost or LightGBM for state-of-the-art results
